In [15]:
import pymysql

connection = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=input("Enter MySQL password: "),
    database="bingeplay"
)

print("Connected successfully to:", connection.get_host_info())

connection.close()

Enter MySQL password:  @chetanff012


Connected successfully to: socket 127.0.0.1:3306


In [16]:
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

password = getpass("Enter MySQL password: ")

url = URL.create(
    "mysql+pymysql",
    username="root",
    password=password,
    host="127.0.0.1",
    port=3306,
    database="bingeplay"
)

engine = create_engine(url)

with engine.connect() as conn:
    print("Connected successfully to bingeplay!")

Enter MySQL password:  ········


Connected successfully to bingeplay!


In [17]:
query = """
SELECT
    COUNT(*) AS active_subscriptions,
    SUM(monthly_price_inr) AS total_monthly_revenue_inr
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""

q1 = pd.read_sql(query, engine)
print(q1.to_string(index=False))

 active_subscriptions  total_monthly_revenue_inr
                 2340                   784260.0


## Q2 — Signup Momentum
**Question:** Count new user signups for each month from January to June 2024 and identify the month with the highest number of signups.

In [18]:
query = """
SELECT
    MONTH(signup_date) AS month,
    COUNT(*) AS signup_count
FROM users
WHERE signup_date >= '2024-01-01'
  AND signup_date < '2024-07-01'
GROUP BY MONTH(signup_date)
ORDER BY month;
"""

q2 = pd.read_sql(query, engine)
print(q2.to_string(index=False))

month_names = {
    1: "January", 2: "February", 3: "March",
    4: "April", 5: "May", 6: "June"
}

top_month = q2.loc[q2["signup_count"].idxmax()]

print(
    "\nHighest signup month:",
    month_names[int(top_month["month"])],
    "-",
    int(top_month["signup_count"]),
    "signups"
)

 month  signup_count
     1           350
     2           400
     3           500
     4           550
     5           600
     6           600

Highest signup month: May - 600 signups


## Q3 — Device Analytics

**Question:** For each device type, calculate the total sessions, total watch minutes, average watch minutes per session, and completion rate. Exclude sessions with a NULL user_id. Round the completion rate to two decimal places.

In [19]:
query = """
SELECT
    device_type,
    COUNT(*) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS completion_rate_percent
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY device_type;
"""

q3 = pd.read_sql(query, engine)
print(q3.to_string(index=False))

device_type  total_sessions  total_watch_minutes  avg_watch_minutes  completion_rate_percent
     Laptop           15105             453434.0              30.02                    60.51
     Mobile           50172            1504355.0              29.98                    60.24
     Tablet            7091             210733.0              29.72                    59.79
         TV           27981             840595.0              30.04                    59.98


## Q4 — Rating Distribution

**Question:** Calculate the count and percentage of ratings for each star value from 1 to 5, and calculate the percentage of all ratings that are 4 or 5 stars.

In [20]:
query = """
SELECT
    stars,
    COUNT(*) AS rating_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM ratings), 2)
        AS rating_percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""

q4 = pd.read_sql(query, engine)
print(q4.to_string(index=False))

query_4_5 = """
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN stars IN (4, 5) THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS percentage_4_or_5_star_ratings
FROM ratings;
"""

q4_summary = pd.read_sql(query_4_5, engine)
print("\nPercentage of 4- or 5-star ratings:")
print(q4_summary.to_string(index=False))

 stars  rating_count  rating_percentage
     1           234               4.68
     2           352               7.04
     3           847              16.94
     4          1781              35.62
     5          1786              35.72

Percentage of 4- or 5-star ratings:
 percentage_4_or_5_star_ratings
                          71.34


## Q5 — Originals vs Acquired Content

**Question:** Compare BingePlay Originals (`is_original = 1`) with acquired content (`is_original = 0`). Calculate the number of shows, average IMDb rating, and average release year for each group. Explain which group performs better on ratings and by how much.

In [21]:
query = """
SELECT
    CASE
        WHEN is_original = 1 THEN 'BingePlay Originals'
        ELSE 'Acquired Content'
    END AS content_type,
    COUNT(*) AS number_of_shows,
    ROUND(AVG(imdb_rating), 2) AS average_imdb_rating,
    ROUND(AVG(release_year), 2) AS average_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;
"""

q5 = pd.read_sql(query, engine)
print(q5.to_string(index=False))

original_rating = q5.loc[
    q5["content_type"] == "BingePlay Originals",
    "average_imdb_rating"
].iloc[0]

acquired_rating = q5.loc[
    q5["content_type"] == "Acquired Content",
    "average_imdb_rating"
].iloc[0]

if original_rating > acquired_rating:
    print(f"\nBingePlay Originals perform better by {original_rating - acquired_rating:.2f} rating points.")
elif acquired_rating > original_rating:
    print(f"\nAcquired Content performs better by {acquired_rating - original_rating:.2f} rating points.")
else:
    print("\nBoth groups have the same average IMDb rating.")

       content_type  number_of_shows  average_imdb_rating  average_release_year
BingePlay Originals               30                 7.92               2020.37
   Acquired Content               70                 6.63               2020.73

BingePlay Originals perform better by 1.29 rating points.


## Q6 — Binge Day Detection

**Question:** For Q2 2024 (1 April to 30 June), identify binge days where the same user watched the same show at least 5 times on the same date. Report the total binge days, the user with the most binge days, and that user's binge-day count.

In [22]:
query = """
WITH binge_days AS (
    SELECT
        user_id,
        show_id,
        session_date,
        COUNT(*) AS sessions_on_day
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date >= '2024-04-01'
      AND session_date < '2024-07-01'
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
),
user_binge_counts AS (
    SELECT
        user_id,
        COUNT(*) AS binge_day_count
    FROM binge_days
    GROUP BY user_id
)
SELECT
    (SELECT COUNT(*) FROM binge_days) AS total_binge_days,
    (SELECT user_id
     FROM user_binge_counts
     ORDER BY binge_day_count DESC, user_id
     LIMIT 1) AS top_user_id,
    (SELECT MAX(binge_day_count)
     FROM user_binge_counts) AS top_user_binge_days;
"""

q6 = pd.read_sql(query, engine)
print(q6.to_string(index=False))

 total_binge_days top_user_id  top_user_binge_days
              414      U02956                    8


## Q7 — Q1 Signups Who Never Watched

**Question:** Find users who signed up between January and March 2024 and never watched a single session. Report the total Q1 signups and the number who never watched anything.

In [23]:
query = """
SELECT
    COUNT(*) AS total_q1_signups,
    SUM(
        CASE
            WHEN NOT EXISTS (
                SELECT 1
                FROM watch_sessions ws
                WHERE ws.user_id = u.user_id
            )
            THEN 1 ELSE 0
        END
    ) AS q1_signups_never_watched
FROM users u
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-04-01';
"""

q7 = pd.read_sql(query, engine)
print(q7.to_string(index=False))

 total_q1_signups  q1_signups_never_watched
             1250                     226.0


## Q8 — Overpaying Premium/Family Users

**Question:** Find users whose current plan as of 30 June 2024 is Premium or Family, but whose entire watch history consists only of shows available on the Basic plan.

**Goal:** Count users who could be candidates for a downgrade offer.

In [24]:
query = """
WITH latest_active AS (
    SELECT
        user_id,
        plan,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)
SELECT COUNT(*) AS overpaying_users
FROM latest_active la
WHERE la.rn = 1
  AND la.plan IN ('Premium', 'Family')
  AND EXISTS (
      SELECT 1
      FROM watch_sessions ws
      WHERE ws.user_id = la.user_id
  )
  AND NOT EXISTS (
      SELECT 1
      FROM watch_sessions ws
      JOIN shows s ON s.show_id = ws.show_id
      WHERE ws.user_id = la.user_id
        AND s.min_plan IN ('Premium', 'Family')
  );
"""

q8 = pd.read_sql(query, engine)
print(q8.to_string(index=False))

 overpaying_users
                7


    ## Q9 — Upgrade Success Cohort

**Question:** Find users who signed up in January 2024, started with a Basic subscription, later upgraded to Premium or Family, and remained active as of 30 June 2024.

**Deliverables:** Number of qualifying users and their average days from signup to their first upgrade.

In [25]:
query = """
WITH ordered_subscriptions AS (
    SELECT
        user_id,
        plan,
        start_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date, subscription_id
        ) AS rn
    FROM subscriptions
),
first_plan AS (
    SELECT user_id, plan
    FROM ordered_subscriptions
    WHERE rn = 1
),
first_upgrade AS (
    SELECT
        s.user_id,
        MIN(s.start_date) AS first_upgrade_date
    FROM subscriptions s
    JOIN first_plan fp ON fp.user_id = s.user_id
    WHERE fp.plan = 'Basic'
      AND s.plan IN ('Premium', 'Family')
      AND s.start_date > (
          SELECT MIN(s2.start_date)
          FROM subscriptions s2
          WHERE s2.user_id = s.user_id
      )
    GROUP BY s.user_id
),
active_users AS (
    SELECT DISTINCT user_id
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)
SELECT
    COUNT(*) AS upgrade_users,
    ROUND(AVG(DATEDIFF(fu.first_upgrade_date, u.signup_date)), 2)
        AS avg_days_to_upgrade
FROM first_upgrade fu
JOIN users u ON u.user_id = fu.user_id
JOIN active_users au ON au.user_id = fu.user_id
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-02-01';
"""

q9 = pd.read_sql(query, engine)
print(q9.to_string(index=False))

 upgrade_users  avg_days_to_upgrade
            55                64.96


## Q10 — Cliffhanger Comebacks

**Question:** A cliffhanger comeback occurs when a user has an incomplete session for a show and watches the same show again 1–7 days later.

**Deliverables:** Total unique comeback events, the show ID with the most comebacks, and its title.

In [26]:
query = """
WITH comeback_events AS (
    SELECT DISTINCT
        a.user_id,
        a.show_id,
        a.session_date AS incomplete_date
    FROM watch_sessions a
    JOIN watch_sessions b
      ON a.user_id = b.user_id
     AND a.show_id = b.show_id
     AND b.session_date BETWEEN
         DATE_ADD(a.session_date, INTERVAL 1 DAY)
         AND DATE_ADD(a.session_date, INTERVAL 7 DAY)
    WHERE a.completed = 0
      AND a.user_id IS NOT NULL
),
show_comebacks AS (
    SELECT
        ce.show_id,
        s.title,
        COUNT(*) AS comeback_count
    FROM comeback_events ce
    JOIN shows s ON s.show_id = ce.show_id
    GROUP BY ce.show_id, s.title
),
top_show AS (
    SELECT show_id, title, comeback_count
    FROM show_comebacks
    ORDER BY comeback_count DESC, show_id
    LIMIT 1
)
SELECT
    (SELECT COUNT(*) FROM comeback_events) AS total_comeback_events,
    show_id AS top_show_id,
    title AS top_show_title,
    comeback_count AS top_show_comebacks
FROM top_show;
"""

q10 = pd.read_sql(query, engine)
print(q10.to_string(index=False))

 total_comeback_events top_show_id   top_show_title  top_show_comebacks
                  4345        S088 Rayalaseema Raga                  64


## Q11 — Consecutive-Week Engagement

**Question:** Find users who watched at least one session in four or more consecutive calendar weeks, using ISO week numbering.

**Deliverables:** Number of users with a streak of at least four weeks, the longest streak, and one user ID with that longest streak.

In [27]:
query = """
WITH distinct_weeks AS (
    SELECT DISTINCT
        user_id,
        YEARWEEK(session_date, 3) AS week_key
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
numbered_weeks AS (
    SELECT
        user_id,
        week_key,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY week_key
        ) AS rn
    FROM distinct_weeks
),
streaks AS (
    SELECT
        user_id,
        week_key - rn AS grp,
        COUNT(*) AS streak_weeks
    FROM numbered_weeks
    GROUP BY user_id, week_key - rn
),
long_streaks AS (
    SELECT
        user_id,
        MAX(streak_weeks) AS longest_streak
    FROM streaks
    GROUP BY user_id
)
SELECT
    SUM(CASE WHEN longest_streak >= 4 THEN 1 ELSE 0 END)
        AS users_with_4plus_week_streak,
    MAX(longest_streak) AS longest_streak_weeks,
    (
        SELECT user_id
        FROM long_streaks
        ORDER BY longest_streak DESC, user_id
        LIMIT 1
    ) AS longest_streak_user
FROM long_streaks;
"""

q11 = pd.read_sql(query, engine)
print(q11.to_string(index=False))

 users_with_4plus_week_streak  longest_streak_weeks longest_streak_user
                       1675.0                    26              U00213


## Q12 — Churn Signal Detection

**Question:** Identify users whose total watch minutes in June 2024 dropped by 50% or more compared with May 2024.

**Deliverables:** User ID, name, May watch minutes, June watch minutes, drop percentage rounded to two decimal places, and the total number of churn-signal users.

In [28]:
query = """
WITH monthly_watch AS (
    SELECT
        user_id,
        SUM(
            CASE
                WHEN session_date >= '2024-05-01'
                 AND session_date < '2024-06-01'
                THEN watch_minutes ELSE 0
            END
        ) AS may_watch_minutes,
        SUM(
            CASE
                WHEN session_date >= '2024-06-01'
                 AND session_date < '2024-07-01'
                THEN watch_minutes ELSE 0
            END
        ) AS june_watch_minutes
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date >= '2024-05-01'
      AND session_date < '2024-07-01'
    GROUP BY user_id
),
churn_signals AS (
    SELECT
        mw.user_id,
        u.name,
        mw.may_watch_minutes,
        mw.june_watch_minutes,
        ROUND(
            100.0 * (mw.may_watch_minutes - mw.june_watch_minutes)
            / mw.may_watch_minutes,
            2
        ) AS drop_percentage
    FROM monthly_watch mw
    JOIN users u ON u.user_id = mw.user_id
    WHERE mw.may_watch_minutes > 0
      AND mw.june_watch_minutes <= mw.may_watch_minutes * 0.50
)
SELECT
    user_id,
    name,
    may_watch_minutes,
    june_watch_minutes,
    drop_percentage,
    COUNT(*) OVER () AS total_churn_signal_users
FROM churn_signals
ORDER BY drop_percentage DESC, user_id;
"""

q12 = pd.read_sql(query, engine)
print(q12.to_string(index=False))
print("\nTotal churn-signal users:", len(q12))

user_id                name  may_watch_minutes  june_watch_minutes  drop_percentage  total_churn_signal_users
 U00023      Amit Mukherjee               43.0                 0.0           100.00                       521
 U00166    Shaurya Malhotra               94.0                 0.0           100.00                       521
 U00211          Ravi Menon              336.0                 0.0           100.00                       521
 U00225       Kritika Patil              209.0                 0.0           100.00                       521
 U00237      Shaurya Bansal              126.0                 0.0           100.00                       521
 U00262          Suresh Roy              158.0                 0.0           100.00                       521
 U00271         Krish Patel              227.0                 0.0           100.00                       521
 U00282          Tara Kumar               47.0                 0.0           100.00                       521
 U00289   